In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [3]:
train_dataset = datasets.ImageFolder(
    "../../datasets/hymenoptera_data/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "../../datasets/hymenoptera_data/val",
    transform=val_transform
)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [5]:
print(train_dataset.classes)
print(train_dataset.class_to_idx)

print(len(train_dataset))
print(len(val_dataset))

['ants', 'bees']
{'ants': 0, 'bees': 1}
244
153


In [6]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(
    model.fc.in_features,
    2
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
print(device)

cpu


In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/5 Loss: 0.6709
Epoch 2/5 Loss: 0.5790
Epoch 3/5 Loss: 0.4697
Epoch 4/5 Loss: 0.3947
Epoch 5/5 Loss: 0.3662


In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        correct += (predicted == labels).sum().item()

        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 88.24%


In [10]:
torch.save(
    model.state_dict(),
    "../../models/resnet18_ants_bees.pth"
)